<a href="https://colab.research.google.com/github/Giocrisrai/aprendizaje-refuerzo-duoc/blob/main/notebooks/07_programacion_genetica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 07 · Programación genética
**Machine Learning — Duoc UC**

Ahora el individuo no es una lista de bits. Es un árbol que se puede evaluar: hojas con números del 1 al 9, nodos internos con `+`, `−` o `×`.

```
      ×
     / \
    6   +
       / \
      9   3
```

Eso es `(6 × (9 + 3))`, o sea 72. Cruzar es cambiar un subárbol por el de otro individuo. Mutar es reemplazar un subárbol por uno nuevo.

La meta de la demo es una expresión que dé **459**. La aptitud premia estar cerca y, con un empate, el árbol más corto.


In [1]:
#@title ⚙️ Configuración (ejecutar primero — en Colab clona el repositorio, sin subir archivos)
import os, sys, subprocess, glob

REPO_URL = "https://github.com/Giocrisrai/aprendizaje-refuerzo-duoc"
EN_COLAB = "google.colab" in sys.modules
ARCHIVOS = ("rl_utils.py", "salmon_env.py", "grilla.py", "evolucion.py")
DATOS = (
    "../data/runs_cliffwalking.csv",
    "../data/historial_inventario.csv",
    "../data/historial_operador_salmon.csv",
)

def buscar_notebooks():
    """Carpeta que contiene los .py de la clase, o None."""
    if all(os.path.exists(nombre) for nombre in ARCHIVOS):
        return os.getcwd()
    candidatos = [
        "/content/aprendizaje-refuerzo-duoc/notebooks",
        "/content/rl_aprendizaje_refuerzo/notebooks",
    ]
    candidatos += glob.glob("/content/**/notebooks", recursive=True)
    vistos = []
    for carpeta in candidatos:
        if carpeta in vistos or not os.path.isdir(carpeta):
            continue
        vistos.append(carpeta)
        if all(os.path.exists(os.path.join(carpeta, nombre)) for nombre in ARCHIVOS):
            return carpeta
    return None

if EN_COLAB and buscar_notebooks() is None:
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/aprendizaje-refuerzo-duoc"],
        check=False,
    )

carpeta = buscar_notebooks()
if carpeta is None:
    raise FileNotFoundError(
        "No encuentro " + ", ".join(ARCHIVOS) + ". "
        "En Colab esta celda clona el repositorio: ahí están los módulos y data/. "
        "En el computador, abre el notebook desde la carpeta notebooks/."
    )
os.chdir(carpeta)

if EN_COLAB:
    req = os.path.abspath(os.path.join(os.getcwd(), "..", "requirements.txt"))
    if os.path.exists(req):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req])
try:
    import numpy, pandas, matplotlib
except ImportError as exc:
    raise ImportError("Faltan numpy, pandas o matplotlib. En local, desde la raíz del repositorio: uv sync --group dev") from exc
faltan = [ruta for ruta in DATOS if not os.path.exists(ruta)]
if faltan:
    raise FileNotFoundError("Faltan datos del repositorio: " + ", ".join(faltan))
print("✅ Listo. Carpeta de trabajo:", os.getcwd(), "| Colab" if EN_COLAB else "| local")
print("Librerías:", "numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)


✅ Listo. Carpeta de trabajo: /Users/giocrisraigodoy/Documents/DUOC/2026-2/rl_clase/notebooks | local


In [2]:
from evolucion import arbol_aleatorio, evaluar, programacion_genetica, texto_arbol
import numpy as np

rng = np.random.default_rng(0)
ejemplo = arbol_aleatorio(rng, profundidad=2, terminales=range(1, 10))
print(texto_arbol(ejemplo), "=", evaluar(ejemplo))

arbol, valor, historia = programacion_genetica(semilla=0)
print("encontró:", texto_arbol(arbol), "=", valor)
print("error inicial de la mejor:", historia[0], "→ final:", historia[-1])


(3 - 2) = 1.0
encontró: (9 * (5 - (((2 - 6) + (((2 * 8) * (1 - 2)) - (((3 * 3) + 8) + 6))) - ((5 - 4) * (7 - 4))))) = 459.0
error inicial de la mejor: 93.0 → final: 0.0


## Ejercicio
`error_expresion` es el valor absoluto entre lo que da el árbol y el objetivo. Sin el castigo por tamaño: solo el error.


In [3]:
def error_expresion(arbol, objetivo):
    raise NotImplementedError("completa esta función")


In [4]:
from evolucion import evaluar

hoja = ("n", 5)  # el árbol que solo vale 5
try:
    assert error_expresion(hoja, 12) == 7
    assert error_expresion(("n", 459), 459) == 0
    print("✅ error(5, 12) =", error_expresion(hoja, 12))
except NotImplementedError:
    print("Pendiente: completa error_expresion. evaluar(arbol) ya está importado.")
except AssertionError:
    print("La hoja 5 contra el objetivo 12 está a 7. Usa evaluar(arbol).")


Pendiente: completa error_expresion. evaluar(arbol) ya está importado.


In [5]:
#@title 💡 Solución ejercicio (doble clic para ver el código)
from evolucion import evaluar

def error_expresion(arbol, objetivo):
    return abs(evaluar(arbol) - objetivo)


## Para qué sirve esto en el desafío
Una política puede ser una fórmula (`si el día pasa de 40, cosechar`) en vez de una tabla. Programación genética busca esa fórmula. En el centro de salmón, con el estado ya diseñado, la tabla del notebook 03 alcanza y se puede leer en la defensa. Un árbol de veinte nodos que nadie puede explicar se parece a un acierto y no lo es.

Sigue en `08_neuroevolucion.ipynb` si quieren ver la misma idea con una red.

### Qué es ejemplo y qué es tarea
La expresión que da 459 se muestra en clase. `error_expresion` lo escriben ustedes.

En pocos minutos, la diferencia entre una lista de bits y un árbol que es un programa: [algoritmo genético, programación genética y programación evolutiva](https://www.youtube.com/watch?v=u8O82yW1z7w).
